# Walkthrough فنی دفاع پایان‌نامه

این notebook برای تمرین و نمایش فنی پایان‌نامه ساخته شده است. هدفش این نیست که در جلسه همهٔ مدل‌ها را از صفر train کند؛ هدف این است که داده، خروجی‌ها، عددهای اصلی و منطق نتیجه‌گیری را شفاف و قابل توضیح نشان بدهد.

**مسیر امن روز دفاع:** اگر وقت کم است، فقط `python scripts/defense_demo.py` را اجرا کن. این notebook برای تمرین و توضیح مرحله‌به‌مرحله بهتر است.

## 1. سؤال پژوهش و ادعای اصلی

سؤال اصلی پایان‌نامه:

> آیا می‌توان با دادهٔ محدود، حساس و غیرمتمرکز ICU، امکان‌پذیری درمان/ترخیص را با ترکیب مدل‌های کلاسیک، نمایش توکنی، few-shot learning و یادگیری فدرال پیش‌بینی کرد؟

ادعای دقیق و دفاع‌پذیر:

> few-shot خام روی دادهٔ جدولی بالینی از Random Forest ضعیف‌تر بود؛ اما در pipeline ترکیبی/stacked، فاصله تا سقف جدولی تقریباً بسته شد.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    display = print

# این cell تلاش می‌کند ریشهٔ پروژه را پیدا کند، حتی اگر notebook را از داخل پوشهٔ notebooks باز کرده باشی.
ROOT = Path.cwd()
if not (ROOT / "scripts" / "defense_demo.py").exists():
    if (ROOT.parent / "scripts" / "defense_demo.py").exists():
        ROOT = ROOT.parent
    else:
        ROOT = Path("/Users/moe/Programming/Thesis-curser")
os.chdir(ROOT)

TABLES = ROOT / "results" / "tables"
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "processed"

print("Project root:", ROOT)
print("Python:", sys.executable)

## 2. دیتاست نهایی چه شکلی است؟

اینجا `thesis_dataset.parquet` را می‌خوانیم. این همان دیتاست جدولی نهایی است که آزمایش‌های اصلی روی آن انجام شده‌اند. ستون `feasible` برچسب دودویی است؛ یعنی خروجی هدف مدل.

In [ ]:
df = pd.read_parquet(DATA / "thesis_dataset.parquet")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

print("\nTarget distribution (feasible):")
display(df["feasible"].value_counts(dropna=False).rename("count").to_frame())

print("\nProtocol / disease-node distribution:")
display(df["protocol"].value_counts(dropna=False).sort_index().rename("count").to_frame())

## 3. نمایش توکنی و گره‌های فدرال

`tokens.npy` نمایش عددی/توکنی بیماران است. فایل‌های `node_*.parquet` هم داده را به چند گره برای شبیه‌سازی federated learning تقسیم می‌کنند.

In [ ]:
tokens = np.load(DATA / "tokens.npy")
node_files = sorted(DATA.glob("node_*.parquet"))
node_sizes = {p.name: len(pd.read_parquet(p)) for p in node_files}

print("Token tensor shape:", tokens.shape)
print("Federated node sizes:")
display(pd.Series(node_sizes, name="rows").to_frame())

## 4. اجرای امن دفاع

این همان اسکریپتی است که روز دفاع باید اجرا کنی. اسکریپت مدل‌ها را از صفر train نمی‌کند؛ خروجی‌های frozen را می‌خواند، فایل‌های ضروری را check می‌کند، و عددهای اصلی را چاپ می‌کند.

In [ ]:
result = subprocess.run(
    [sys.executable, "scripts/defense_demo.py"],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)
print("Return code:", result.returncode)

## 5. مقایسهٔ سخت‌گیرانهٔ Exp1

این جدول مهم‌ترین جا برای صداقت علمی است. در این پروتکل، `RandomForest` از `Proposed_FewShot` بهتر است. پس نباید بگویی few-shot خام بهتر از مدل کلاسیک است.

In [ ]:
exp1 = pd.read_csv(TABLES / "exp1_results.csv")
display(exp1.sort_values("auroc", ascending=False))

plot_df = exp1[["model", "auroc"]].sort_values("auroc", ascending=True)
ax = plot_df.plot.barh(x="model", y="auroc", legend=False, figsize=(8, 4), color="#2f6f73")
ax.set_title("Experiment 1: Strict AUROC Comparison")
ax.set_xlabel("AUROC")
ax.set_xlim(0.5, max(0.8, plot_df["auroc"].max() + 0.03))
plt.show()

## 6. اثر تعداد shotها

این بخش نشان می‌دهد صرفاً زیاد کردن K معجزه نمی‌کند. در این دیتاست، کیفیت representation و ترکیب با مدل کلاسیک مهم‌تر از افزایش سادهٔ تعداد نمونه‌های پشتیبان است.

In [ ]:
exp2 = pd.read_csv(TABLES / "exp2_results.csv")
display(exp2)

ax = exp2.plot(x="k", y="auroc", yerr="auroc_std", marker="o", capsize=4, figsize=(7, 4), color="#9a5b2f")
ax.set_title("Experiment 2: K-shot Sensitivity")
ax.set_xlabel("K shots")
ax.set_ylabel("AUROC")
plt.show()

## 7. یادگیری فدرال

اینجا centralized و federated مقایسه می‌شوند. پیام اصلی: در این شبیه‌سازی، افت AUROC کوچک است، اما این هنوز استقرار واقعی چندمرکزی نیست.

In [ ]:
exp3 = pd.read_csv(TABLES / "exp3_results.csv")
display(exp3)

fed = exp3[exp3["setup"].isin(["Centralized", "Federated"])]
ax = fed.plot.bar(x="setup", y="auroc", legend=False, figsize=(6, 4), color=["#2f6f73", "#c7772e"])
ax.set_title("Experiment 3: Centralized vs Federated")
ax.set_ylabel("AUROC")
plt.xticks(rotation=0)
plt.show()

## 8. نتیجهٔ نهایی pipeline ترکیبی

`exp_fewshot_best.json` خروجی مهم برای پیام نهایی است: few-shot تنها کافی نیست، اما stacked/fusion pipeline به سقف جدولی نزدیک می‌شود.

In [ ]:
with open(RESULTS / "exp_fewshot_best.json") as f:
    best = json.load(f)

fewshot = best["fewshot_best"]
stacked = best["stacked_best"]
final_best = pd.read_csv(TABLES / "exp_final_best_model.csv")

summary = pd.DataFrame([
    {"level": "Few-shot-only BEST", "auroc": fewshot["auroc"], "ci_low": fewshot["auroc_ci_lo"], "ci_high": fewshot["auroc_ci_hi"]},
    {"level": "Stacked BEST", "auroc": stacked["auroc"], "ci_low": stacked["auroc_ci_lo"], "ci_high": stacked["auroc_ci_hi"]},
    {"level": "Tabular ceiling/final reference", "auroc": float(final_best["auroc"].max()), "ci_low": np.nan, "ci_high": np.nan},
])
display(summary)

ax = summary.plot.bar(x="level", y="auroc", legend=False, figsize=(8, 4), color="#496a9a")
ax.set_title("Final Thesis Claim: Fusion Closes the Gap")
ax.set_ylabel("AUROC")
ax.set_ylim(0.55, 0.80)
plt.xticks(rotation=20, ha="right")
plt.show()

## 9. جمع‌بندی که باید شفاهی بگویی

اگر خواستی خیلی ساده توضیح بدهی:

1. من دادهٔ ICU را به دیتاست جدولی، توکن‌های علائم، و گره‌های فدرال تبدیل کردم.
2. مدل‌های کلاسیک را به‌عنوان baseline قوی اجرا کردم.
3. few-shot خام را تست کردم و دیدم از RF ضعیف‌تر است؛ این را پنهان نمی‌کنم.
4. سپس embedding/few-shot را با مدل کلاسیک در stacked pipeline ترکیب کردم.
5. نتیجهٔ نهایی به AUROC حدود 0.746 رسید و به سقف جدولی حدود 0.747 نزدیک شد.

**جملهٔ طلایی:** ارزش پایان‌نامه فقط یک عدد بالاتر نیست؛ ارزش آن این است که نشان می‌دهد few-shot خام برای دادهٔ جدولی بالینی کافی نیست و باید با representation و مدل کلاسیک ترکیب شود.